# Notebook 1b: Wayback Machine Enrichment
**Project:** Where Do Fitness Businesses Thrive in Utah?

Enriches `data/utah_fitness_v2.csv` with business longevity data by querying the
Internet Archive's CDX API to find the **earliest Wayback Machine snapshot** of each
business's Yelp page.

**Logic:** The first time the Wayback Machine archived a Yelp page is a reasonable
lower bound for how long that business has been listed — and by extension, operating.

**Output fields added:**
- `first_seen_year` — year of the earliest Wayback snapshot (e.g. 2013)
- `years_on_yelp` — `2026 - first_seen_year` (longevity proxy)

**Caveats documented:**
- Wayback Machine doesn't crawl every page immediately — `first_seen_year` is a lower
  bound, not exact founding date.
- ~10–20% of businesses may have no snapshot; these are imputed with the median.
- Saves enriched CSV to `data/utah_fitness_v2.csv` (overwrites in place).

**Runtime:** ~20–25 minutes for ~1,000 businesses at 0.5s/request.

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import requests
import time

CURRENT_YEAR = 2026
CDX_URL = 'http://web.archive.org/cdx/search/cdx'

df = pd.read_csv('data/utah_fitness_v2.csv', dtype={'zip_code': str})
print(f'Loaded {len(df)} businesses.')
print(f'Columns: {list(df.columns)}')

## 1. Wayback CDX Lookup

For each business, we request the single earliest snapshot of its Yelp page.
The CDX API returns a JSON array; the timestamp field is `YYYYMMDDHHmmss`.

In [ ]:
def get_first_wayback_year(biz_id, retries=2):
    """
    Returns the year of the earliest Wayback snapshot for a Yelp business page,
    or None if no snapshot exists.
    """
    params = {
        'url': f'yelp.com/biz/{biz_id}',
        'output': 'json',
        'limit': 1,          # only the oldest
        'fl': 'timestamp',   # only need the date
        'fastLatest': 'false',
        'filter': 'statuscode:200',  # skip redirect/error snapshots
    }
    for attempt in range(retries + 1):
        try:
            resp = requests.get(CDX_URL, params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                # data[0] is the header row ['timestamp'], data[1] is first result
                if len(data) >= 2:
                    timestamp = data[1][0]  # e.g. '20130415123045'
                    return int(timestamp[:4])  # extract year
                return None
            else:
                time.sleep(1)
        except Exception as e:
            if attempt < retries:
                time.sleep(2)
            else:
                return None
    return None


print('Lookup function defined.')
# Quick sanity test on one known business
test_id = df['id'].iloc[0]
test_year = get_first_wayback_year(test_id)
print(f'Test biz_id: {test_id}')
print(f'First seen: {test_year}')

In [ ]:
# ── Main loop ──────────────────────────────────────────────────────────────────
# Runs ~20-25 minutes for 1,000+ businesses.
# Progress printed every 50 businesses.

biz_ids = df['id'].tolist()
first_seen_years = []
no_snapshot_count = 0

print(f'Querying Wayback Machine for {len(biz_ids)} businesses...')
print('(~0.5s per request, expect ~20 minutes total)\n')

for i, biz_id in enumerate(biz_ids):
    year = get_first_wayback_year(biz_id)
    first_seen_years.append(year)
    if year is None:
        no_snapshot_count += 1

    if (i + 1) % 50 == 0:
        found = i + 1 - no_snapshot_count
        print(f'  {i+1}/{len(biz_ids)} queried | '
              f'found: {found} | no snapshot: {no_snapshot_count}')

    time.sleep(0.5)

df['first_seen_year'] = first_seen_years

print(f'\nDone.')
print(f'Businesses with snapshot:    {df["first_seen_year"].notna().sum()}')
print(f'Businesses without snapshot: {df["first_seen_year"].isna().sum()}')
print(f'\nYear distribution:')
print(df['first_seen_year'].value_counts().sort_index())

## 2. Derive `years_on_yelp`

In [ ]:
import matplotlib.pyplot as plt

# Impute missing first_seen_year with the median
median_year = df['first_seen_year'].median()
missing_count = df['first_seen_year'].isna().sum()
print(f'Median first_seen_year: {median_year}')
print(f'Imputing {missing_count} missing values with median ({median_year})')

df['first_seen_year'] = df['first_seen_year'].fillna(median_year)

# Clip to reasonable range — anything before 2004 (Yelp founding year) is noise
df['first_seen_year'] = df['first_seen_year'].clip(lower=2004, upper=CURRENT_YEAR)

# Derive longevity
df['years_on_yelp'] = CURRENT_YEAR - df['first_seen_year']

print(f'\nyears_on_yelp stats:')
print(df['years_on_yelp'].describe().round(2))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['first_seen_year'].astype(int).value_counts().sort_index().plot.bar(
    ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('First Wayback Snapshot Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(df['years_on_yelp'], bins=20, color='steelblue', edgecolor='white')
axes[1].set_title('Years on Yelp (Longevity Proxy)')
axes[1].set_xlabel('Years')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('figures/01b_longevity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/01b_longevity_distribution.png')

## 3. Validate — Do Longer-Running Businesses Perform Better?

In [ ]:
# If years_on_yelp is a meaningful longevity signal, we'd expect older businesses
# to have more reviews and (possibly) higher ratings.

rated = df[df['rating'] > 0].copy()

corr_reviews = rated['years_on_yelp'].corr(rated['review_count'])
corr_rating  = rated['years_on_yelp'].corr(rated['rating'])
corr_success = rated['years_on_yelp'].corr(rated['success_score'])

print('Correlations with years_on_yelp:')
print(f'  review_count:  r = {corr_reviews:.3f}')
print(f'  rating:        r = {corr_rating:.3f}')
print(f'  success_score: r = {corr_success:.3f}')

# Bin by era and look at avg success score
bins = [0, 3, 6, 9, 12, 22]
labels = ['0–3 yrs', '3–6 yrs', '6–9 yrs', '9–12 yrs', '12+ yrs']
rated['age_bucket'] = pd.cut(rated['years_on_yelp'], bins=bins, labels=labels)

bucket_summary = (
    rated.groupby('age_bucket', observed=True)[['review_count', 'rating', 'success_score']]
    .mean()
    .round(3)
)
print('\nAverage metrics by longevity bucket:')
print(bucket_summary)

## 4. Drop `has_hours` and Save Enriched CSV

In [ ]:
# Drop has_hours — years_on_yelp is a cleaner longevity signal
if 'has_hours' in df.columns:
    df = df.drop(columns=['has_hours'])
    print('Dropped has_hours column.')

df.to_csv('data/utah_fitness_v2.csv', index=False)
print(f'Saved enriched dataset: {len(df)} rows, {len(df.columns)} columns')
print(f'New columns: first_seen_year, years_on_yelp')
df[['name', 'rating', 'review_count', 'first_seen_year', 'years_on_yelp']].head(10)